<a href="https://colab.research.google.com/github/trainocate-japan/developing-agentic-ai-with-langchain/blob/main/chap02/exercise/solution/chap02_exercise_2B_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 演習 2-B【正解】: Function Calling 手動 1 周 — ヘルプデスク Step 1

**研修コース「Agentic AI 開発実践 - LangChain 版」/ 第2章「LLM API の基礎」**

この Notebook は演習 2-B の**正解 (solution)** です。
TODO (`# TODO`) を埋める前に答えを見てしまわないよう、**まずは starter で自力で挑戦**してください。
詰まったとき・答え合わせのときにこちらを参照しましょう。

## この Notebook で学ぶこと

第1章の ReAct ループの「行動」を実現する仕組み——**Function Calling**——を、
フレームワークなし (openai パッケージ直接) で手動で 1 周します。題材は本コースの演習ストーリー
「**社内 IT ヘルプデスクエージェント**」の第一歩 (Step 1) として、社内システムの稼働状況に答える
`get_system_status(service)` ツールです。
ハンズオン 2-A の後半 (セクション 7) で `get_weather` を題材に動かして確認した 4 ステップを、
今度は helpdesk のツールで自分の手で組み立てる位置づけです。

Function Calling の 1 周は次の 4 ステップ:

1. **ツールを定義してリクエスト** — `tools` に関数の名前・説明・引数スキーマを添えて送る
2. **tool_calls を受け取る** — `finish_reason="tool_calls"` と呼び出し宣言を受信する
3. **アプリ側で関数を実行** — `arguments` を `json.loads` し、自分のコードで関数を呼ぶ
4. **結果を返して最終応答** — `role:"tool"` のメッセージを履歴に積んで再送信する

> **本節最大のポイント (先に宣言)**: **LLM は関数を実行しません。**
> モデルがするのは「この関数をこの引数で実行したい」という**意図を JSON で宣言する**ことだけ。
> 実際に関数を実行するのは常に**アプリケーション側 (あなたのコード)** です。

## 前提条件

- **ハンズオン 2-A を完了している**こと (同一環境で続けて実施。`client` と `MODEL` の準備が前提。後半の `get_weather` デモで FC の 4 ステップを確認済み)
- Google Colab で開き、Colab シークレットに `OPENAI_API_KEY` を登録済みであること

## 所要時間

約 18 分

---
> **モデル名について**: モデル名は変数 `MODEL` に集約しています (例: `MODEL = "gpt-5.4"`)。
> 研修実施時は講師が指定する最新モデル名に差し替えてください。


## 0. セットアップ

ハンズオン 2-A と同じセットアップです (この Notebook を単体で開いた場合にも動くよう再掲します)。
すでに同一セッションで 2-A を実行済みなら、このセクションは流すだけで構いません。


In [ ]:
# openai パッケージを最新版へ (研修実施時は最新版にピン留め推奨)
!pip install -U openai

In [ ]:
import os
import json

# Colab シークレットから API キーを読み込む。Colab 以外では環境変数をそのまま使う
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass  # Colab 以外では環境変数 OPENAI_API_KEY が設定済みとみなす

from openai import OpenAI

client = OpenAI()          # 環境変数 OPENAI_API_KEY からキーを自動取得
MODEL = "gpt-5.4"          # モデル名は変数に集約 (研修実施時に最新へ差し替え)

print("準備完了。使用モデル:", MODEL)

---

## 1. 題材の関数を用意する — get_system_status

モデルに「呼び出させる」ための関数を用意します。
本物の監視 API は使わず、**固定の稼働状況を返すダミー実装**にします (仕組みの学習にはこれで十分)。
複数の社内サービスの状態を辞書で持ち、`service` 名で引きます。

ハンズオン 2-A の天気エージェント (`get_weather`) を、ヘルプデスク版に置き換えたものです。


In [ ]:
# 社内システムの稼働状況を返すダミー関数。
# 実運用では監視システムの API を叩くが、ここでは固定の辞書から返す (学習用)。
SYSTEM_STATUS = {
    "勤怠システム": "正常稼働中",
    "経費精算システム": "正常稼働中",
    "メールサーバー": "一部遅延あり (調査中)",
    "VPN": "メンテナンス中 (本日 22:00 まで)",
}


def get_system_status(service: str) -> str:
    """指定された社内システムの現在の稼働状況を返す (ダミー実装)"""
    status = SYSTEM_STATUS.get(service, "不明 (登録されていないシステムです)")
    return f"{service}の稼働状況: {status}"


# 動作確認: 関数単体で呼んでみる
print(get_system_status("勤怠システム"))
print(get_system_status("VPN"))
print(get_system_status("存在しないシステム"))

---

## 2. ステップ①: ツールを定義してリクエストする

次に、この関数の存在をモデルに伝えるための**ツール定義**を書きます。
ツール定義は「関数の取扱説明書」を JSON で書いたものです。
`parameters` の部分は **JSON Schema** という標準形式で引数の仕様を記述します。

> **starter の TODO① はここ**: `parameters` (型・`required`・`description`) を埋める部分です。

ここで特に重要なのが **`description`** です。モデルは皆さんの Python コードの中身を見られません。
**ツールを使うか・どのツールを使うかを判断する材料は、この description (と関数名・引数スキーマ) だけ**です。
description の書き方がエージェントの賢さを直接左右します
(第3章では `@tool` 関数の docstring がこの description に変換されることを学びます)。


In [ ]:
# ツール定義: get_system_status の「取扱説明書」を JSON で書く
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_system_status",
            # description はモデルがツールを選ぶ唯一の判断材料。丁寧に書く
            "description": "社内システム (勤怠システム、経費精算システム、メールサーバー、VPN 等) の現在の稼働状況を取得する",
            "parameters": {                  # 引数の仕様を JSON Schema で記述
                "type": "object",
                "properties": {
                    "service": {
                        "type": "string",
                        "description": "稼働状況を知りたい社内システムの名称。例: 勤怠システム",
                    }
                },
                "required": ["service"],     # 必須の引数
            },
        },
    }
]

print("ツール定義を作成しました:", tools[0]["function"]["name"])

定義ができたら、`tools` を添えて質問を送ります。
「**勤怠システムは動いていますか?**」と尋ねてみましょう。

> **古い記法に注意**: 旧 `functions` / `function_call` パラメータは廃止済みです。
> 現行は `tools` / `tool_choice`。`functions=` を使うコードを見たら古い記事と判断してください。

**期待される結果**: モデルは質問に直接答える代わりに、「`get_system_status` を呼びたい」という
**呼び出し宣言**を返します (次のステップで取り出します)。


In [ ]:
# 「勤怠システムは動いていますか?」を tools 付きで送信する
messages = [{"role": "user", "content": "勤怠システムは動いていますか?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,          # ツール定義を添える
)

print("リクエスト送信完了。レスポンスを次のステップで確認します。")

---

## 3. ステップ②: tool_calls を受け取る

モデルが「この質問にはツールが必要だ」と判断すると、レスポンスの様子が一変します。
`finish_reason` が `"tool_calls"` になり、`message.content` の代わりに
`message.tool_calls` に呼び出し宣言が入ります。

> **starter の TODO② はここ**: `message.tool_calls[0]` から関数名・引数を取り出し、
> `arguments` を `json.loads` でパースする部分です。

> **定番のつまずき**: `arguments` は **dict ではなく JSON 文字列**です。
> 見た目は dict そっくりですが、`tool_call.function.arguments["service"]` と書くと TypeError になります。
> 必ず `json.loads()` でパースしてください。


In [ ]:
# finish_reason が "tool_calls" になっていることを確認
print("finish_reason:", response.choices[0].finish_reason)   # => "tool_calls" ("stop" ではない!)

# 呼び出し宣言を取り出す。tool_calls はリストなので [0] で最初の宣言を取る
tool_call = response.choices[0].message.tool_calls[0]

print("tool_call.id        :", tool_call.id)                  # 例: "call_abc123" (④で必要になる ID)
print("function.name       :", tool_call.function.name)       # => "get_system_status"
print("function.arguments  :", tool_call.function.arguments)  # => '{"service": "勤怠システム"}' ← JSON「文字列」
print("arguments の型       :", type(tool_call.function.arguments))  # => <class 'str'> (dict ではない!)

---

## 4. ステップ③: アプリ側で関数を実行する

宣言を読み取ったら、実行するのは**アプリ側の仕事**です。
`arguments` を `json.loads` で dict に変換し、`**args` で関数に渡します。

この 2 行が「**LLM は関数を実行しない**」の動かぬ証拠です。
実行しているのは紛れもなく皆さんの Python コードです。
ここで結果を検査することも、危険なら実行を拒否することも、人間の承認を待つこと
(Human-in-the-Loop、第6章) も、すべてアプリ側の自由です。

**期待される結果**: `get_system_status("勤怠システム")` が実行され、稼働状況の文字列が表示されます。


In [ ]:
# arguments (JSON 文字列) を dict にパースしてから、アプリ側で関数を実行する
args = json.loads(tool_call.function.arguments)   # JSON 文字列 → dict
print("パース後の引数 (dict):", args)

result = get_system_status(**args)                # アプリ側が関数を実行する
print("関数の実行結果       :", result)             # => "勤怠システムの稼働状況: 正常稼働中"


---

## 5. ステップ④: 結果を返して最終応答を得る

最後に、実行結果をモデルに伝えて最終応答をもらいます。
履歴には **2 つ**のメッセージを**正しい順序**で積む必要があります。

1. **(a)** `tool_calls` 入りの assistant メッセージ (= 受け取った `response.choices[0].message` そのもの) を先に積む
2. **(b)** 実行結果を `role:"tool"` で積む。`tool_call_id` でどの呼び出しへの答えかを示す

> **starter の TODO③ はここ**: assistant メッセージ (tool_calls 入り) と tool メッセージの
> 履歴への追加部分です。

> **定番のつまずき 2 つ**:
> - **罠1**: (a) を飛ばして (b) だけ積むと `BadRequestError: messages with role 'tool' must be a
>   response to a preceding message with 'tool_calls'` になる。**履歴は「宣言→結果」のペアで積む**。
> - **罠2**: `tool_call_id` の不一致・返し忘れも `BadRequestError`。**`tool_call_id` はモデルが発行した
>   `tool_calls[0].id` をそのまま使う**。

**期待される結果**: モデルが稼働状況を踏まえた最終応答 (例:「勤怠システムは正常に稼働しています」) を返します。


In [ ]:
# (a) tool_calls 入りの assistant メッセージを先に積む (積み忘れ注意!)
messages.append(response.choices[0].message)

# (b) 実行結果を role:"tool" で積む。tool_call_id でどの呼び出しへの答えかを示す
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,   # ②で受け取った id をそのまま使う
    "content": result,
})

# 全履歴を再送信 → モデルが結果を踏まえた最終応答を生成
final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
print("最終応答:", final.choices[0].message.content)
# => 例: "勤怠システムは正常に稼働しています。" など

これで Function Calling の 1 周が完成しました 🎉

ステートレスな API に対して、「呼び出し宣言」と「実行結果」を履歴として積み増しながら会話を続けている——
ハンズオン 2-A のミニチャットボットと**完全に同じ原理**であることに気付いたでしょうか。

`tool_call_id` は「この結果は、あなたが発行したどの呼び出し宣言への答えか」を対応付ける伝票番号です。


---

## 6. tool_calls が返らないことの確認 (完成の目安)

ツールが必要ない質問では、モデルは `tool_calls` を返さず、普通に `content` で答えます。
「**こんにちは**」を送って確認しましょう。

**期待される結果**: `finish_reason` は `"stop"` (=`"tool_calls"` ではない)、
`tool_calls` は `None`、`content` に挨拶の返事が入ります。
モデルが「これはツール不要」と正しく判断できている証拠です。


In [ ]:
# ツール不要の質問では tool_calls が返らないことを確認する
greeting = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "こんにちは"}],
    tools=tools,          # tools を添えても、不要ならモデルは使わない
)

print("finish_reason:", greeting.choices[0].finish_reason)        # => "stop" ("tool_calls" ではない)
print("tool_calls   :", greeting.choices[0].message.tool_calls)   # => None
print("content      :", greeting.choices[0].message.content)      # => 挨拶の返事

---

## 7. (発展) while ループ化 + ツール 2 つ

ここまではツール呼び出しが 1 回でした。現実のタスクでは、最初のツールの結果を見て次のツールを呼ぶ、
といった連鎖が必要です。そこで、**ステップ 2〜4 を `finish_reason` が `"tool_calls"` でなくなるまで
while ループで回します**。下の疑似コードのイメージを、実際に動くコードにします。

```text
while True:
    response = API へ全履歴を送信 (tools 付き)
    if finish_reason != "tool_calls":
        break                      # ツール不要 → 最終応答が出た
    for tool_call in tool_calls:
        結果 = アプリ側で関数を実行
        履歴に assistant (tool_calls) と tool (結果) を積む
```

ツールも 2 つに増やします: `get_system_status` に加えて、
**メンテナンス予定を返す `get_maintenance_schedule` (ダミー)** を追加します。

> **この while ループこそがエージェントの正体**です。
> 知覚 (履歴とツール結果) → 推論・意思決定 (tool_calls の生成) → 行動 (アプリ側での実行) の反復。
> 第3章で学ぶ `create_agent` は、まさにこのループを自動化してくれます。

まず 2 つ目のツールを定義します。


In [ ]:
# 2 つ目のダミーツール: メンテナンス予定を返す
MAINTENANCE_SCHEDULE = {
    "勤怠システム": "予定なし",
    "経費精算システム": "今週末 (土) 02:00-04:00 に定期メンテナンス",
    "メールサーバー": "予定なし",
    "VPN": "本日 20:00-22:00 にメンテナンス中",
}


def get_maintenance_schedule(service: str) -> str:
    """指定された社内システムのメンテナンス予定を返す (ダミー実装)"""
    schedule = MAINTENANCE_SCHEDULE.get(service, "不明 (登録されていないシステムです)")
    return f"{service}のメンテナンス予定: {schedule}"


# 関数名から実体を引くためのレジストリ (ループ内でモデルが指定した関数を呼び分けるのに使う)
AVAILABLE_FUNCTIONS = {
    "get_system_status": get_system_status,
    "get_maintenance_schedule": get_maintenance_schedule,
}

# 2 ツール分の定義
tools_multi = [
    {
        "type": "function",
        "function": {
            "name": "get_system_status",
            "description": "社内システム (勤怠システム、経費精算システム、メールサーバー、VPN 等) の現在の稼働状況を取得する",
            "parameters": {
                "type": "object",
                "properties": {
                    "service": {"type": "string", "description": "稼働状況を知りたい社内システムの名称。例: 勤怠システム"}
                },
                "required": ["service"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_maintenance_schedule",
            "description": "社内システムのメンテナンス予定 (予定されている停止時間など) を取得する",
            "parameters": {
                "type": "object",
                "properties": {
                    "service": {"type": "string", "description": "メンテナンス予定を知りたい社内システムの名称。例: VPN"}
                },
                "required": ["service"],
            },
        },
    },
]

print("2 つのツールを定義しました:", list(AVAILABLE_FUNCTIONS.keys()))

次に、while ループ本体です。
`finish_reason` が `"tool_calls"` の間はツールを実行して履歴に積み、
`"tool_calls"` でなくなったら (= 最終応答が出たら) ループを抜けます。

`tool_calls` はリストなので `for tool_call in message.tool_calls:` でループ処理します
(parallel tool calls = モデルが複数の呼び出しを一度に返すケースにも対応するため)。

**期待される結果**: 稼働状況とメンテナンス予定の両方を踏まえた最終応答が返ります。
ループが何周回ったか、各周で何のツールが呼ばれたかもログに表示されます。


In [ ]:
# while ループ版の Function Calling (ステップ 2〜4 を繰り返す = エージェントの最小形)
messages = [
    {"role": "system", "content": "あなたは社内 IT ヘルプデスクの一次対応担当です。必要に応じてツールで状況を調べて答えてください。"},
    {"role": "user", "content": "VPN は今使えますか? メンテナンス予定もあわせて教えてください。"},
]

max_turns = 5   # 無限ループ防止の安全弁
for turn in range(max_turns):
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools_multi)
    msg = response.choices[0].message

    # ツールが不要になったら (最終応答が出たら) ループを抜ける
    if response.choices[0].finish_reason != "tool_calls":
        print(f"\n[最終応答] {msg.content}")
        break

    # (a) tool_calls 入りの assistant メッセージを積む
    messages.append(msg)

    # tool_calls はリスト。複数の呼び出し宣言すべてに tool メッセージを返す
    for tool_call in msg.tool_calls:
        func = AVAILABLE_FUNCTIONS[tool_call.function.name]   # 関数名から実体を引く
        args = json.loads(tool_call.function.arguments)       # JSON 文字列 → dict
        result = func(**args)                                 # アプリ側で実行
        print(f"[turn {turn}] {tool_call.function.name}({args}) -> {result}")

        # (b) 実行結果を role:"tool" で積む。tool_call_id を一致させる
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        })
else:
    # for...else: break されずに max_turns に達した場合 (通常は起きない)
    print("\n[警告] 最大ターン数に達しました。")

---

## まとめ — 定番のつまずきチェックリスト

演習で実際にハマりやすいポイントを、最後にもう一度確認しておきましょう。

- [ ] **`arguments` は JSON 文字列**。`json.loads()` でパースしてから使う (dict のように添字アクセスしない)
- [ ] **tool メッセージの前に assistant メッセージ (tool_calls 入り) を積む**。
  順序を逆にしたり (a) を忘れると `BadRequestError`
- [ ] **`tool_call_id` はモデルが発行した `tool_calls[0].id` をそのまま使う**。取り違え・返し忘れも `BadRequestError`
- [ ] **parallel tool calls**: `tool_calls` はリスト。複数返ることがあり、それぞれに tool メッセージを返す
- [ ] **関数を実行するのは誰か?** → **アプリ側 (あなたのコード)**。LLM は JSON で意図を宣言するだけ

### 完成の目安 (達成できたか確認)

- ✅ 「勤怠システムは動いていますか?」への最終応答が得られる (1〜5)
- ✅ 「こんにちは」では `tool_calls` が返らない (6)
- ✅ (発展) while ループ + 2 ツールで連鎖的なツール利用ができる (7)

### この完成コードは保存してください

**完成したコードは必ず保存しておいてください。**
第3章で LangChain の `create_agent` 版と並べて diff を取り、
「フレームワークが何を肩代わりしてくれているのか」を自分のコードで確認します。
手動 1 周を経験した皆さんだからこそ味わえる「ありがたみ」が、そこにあります。

参考までに、本章 (OpenAI API 直叩き) で使った語彙は、第3章の LangChain のメッセージ抽象と次のように 1 対 1 で対応します。

| 本章 (OpenAI API) | 第3章 (LangChain) |
|---|---|
| `{"role": "system", ...}` | `SystemMessage` |
| `{"role": "user", ...}` | `HumanMessage` |
| `{"role": "assistant", ...}` | `AIMessage` |
| `{"role": "tool", ...}` | `ToolMessage` |
| `message.tool_calls` | `AIMessage.tool_calls` |
| `tool_call_id` | `ToolMessage.tool_call_id` |
| ツール定義の `description` | `@tool` 関数の docstring |
| ステップ 2〜4 の while ループ | `create_agent` が自動実行 |
